# Natural Language Processing

In [1]:
import pandas as pd
from blended_learning.config.settings import settings
from blended_learning.utils.decorator import execution_time


In [2]:
df = pd.read_csv(settings.path['output_cluster_path'])

In [3]:
columns = [
    'student_id',
    'open_strengths',
    'open_challenges_suggestions',
    'student_segment',
    'student_segment_label',
    'cluster_label'
]
df_nlp = df[columns].copy()
print(f"NLP dataframe shape: {df_nlp.shape}")
display(df_nlp.head(n=2))

NLP dataframe shape: (420, 6)


,student_id,open_strengths,open_challenges_suggestions,student_segment,student_segment_label,cluster_label
0,e20210528,NaN,NaN,2,Cluster 2: Highly Engaged (Active) Learners,Highly Engaged (Active) Learners
1,e20210686,Nothing,Nothing,1,Cluster 1: Moderately Engaged (Passive) Learners,Moderately Engaged (Passive) Learners


In [4]:
print(f"Before : Length {len(df_nlp)}")

cols = ['open_strengths', 'open_challenges_suggestions']

mask = (
    df_nlp[cols].notna().all(axis=1) &
    (df_nlp[cols].astype(str).apply(lambda x: x.str.strip() != "")).all(axis=1)
)

df_nlp = df_nlp[mask]

print(f"After : Length {len(df_nlp)}")

Before : Length 420
After : Length 408


In [5]:
import re
import pandas as pd

# Regex patterns
KHMER_REGEX = re.compile(r'[\u1780-\u17FF]')
EN_REGEX = re.compile(r'[A-Za-z]')

# 1) Detect language category
def detect_lang_mixed(text):
    if pd.isna(text):
        return "empty"
    
    text = str(text).strip()
    if text == "":
        return "empty"
    
    has_kh = bool(KHMER_REGEX.search(text))
    has_en = bool(EN_REGEX.search(text))
    
    if has_kh and has_en:
        return "mixed"
    elif has_kh:
        return "kh"
    elif has_en:
        return "en"
    else:
        return "other"

# 2) Create language label columns
for col in cols:
    df[f"{col}_lang"] = df[col].apply(detect_lang_mixed)

# 3) Print summary counts + percentages per column
all_langs = []

for col in cols:
    cleaned_mask = df[col].notna() & (df[col].astype(str).str.strip() != "")
    langs = df.loc[cleaned_mask, f"{col}_lang"]
    all_langs.append(langs)
    
    counts = langs.value_counts().reindex(["kh", "mixed", "en", "other"], fill_value=0)
    perc = (counts / counts.sum() * 100).round(1)
    
    summary = pd.DataFrame({
        "count": counts,
        "percent (%)": perc
    })
    
    translation_needed = counts["kh"] + counts["mixed"]
    translation_pct = round((translation_needed / counts.sum()) * 100, 1) if counts.sum() > 0 else 0
    
    print(f"\n{'='*60}")
    print(f"SUMMARY FOR: {col}")
    print(f"{'='*60}")
    print(summary)
    print(f"\nTotal needing translation (kh + mixed): {translation_needed} ({translation_pct}%)")

# 4) Aggregate summary across BOTH text columns
all_langs_combined = pd.concat(all_langs, axis=0)

agg_counts = all_langs_combined.value_counts().reindex(["kh", "mixed", "en", "other"], fill_value=0)
agg_perc = (agg_counts / agg_counts.sum() * 100).round(1)

agg_summary = pd.DataFrame({
    "count": agg_counts,
    "percent (%)": agg_perc
})

agg_translation_needed = agg_counts["kh"] + agg_counts["mixed"]
agg_translation_pct = round((agg_translation_needed / agg_counts.sum()) * 100, 1) if agg_counts.sum() > 0 else 0

print(f"\n{'='*60}")
print("AGGREGATE LANGUAGE SUMMARY (BOTH TEXT COLUMNS)")
print(f"{'='*60}")
print(agg_summary)
print(f"\nTotal needing translation (kh + mixed): {agg_translation_needed} ({agg_translation_pct}%)")
print(f"Total Khmer only: {agg_counts['kh']}")
print(f"Total Mixed: {agg_counts['mixed']}")
print(f"Total Other: {agg_counts['other']}")

# 5) Function to print all responses by language type
def inspect_language(df, col, lang_type):
    print(f"\n\n{'='*80}")
    print(f"{col} → {lang_type.upper()}")
    print(f"{'='*80}\n")
    
    subset = df[df[f"{col}_lang"] == lang_type][col]
    subset = subset.dropna().astype(str).str.strip()
    subset = subset[subset != ""]
    
    if len(subset) == 0:
        print("No responses found.\n")
        return
    
    for i, val in enumerate(subset, 1):
        print(f"{i}. {val}\n")

# 6) Print ALL kh, mixed, and other responses
for col in cols:
    inspect_language(df, col, "kh")
    inspect_language(df, col, "mixed")
    inspect_language(df, col, "other")

# 7) Save all kh, mixed, and other responses to a text file
path = settings.root / "data" / "language_inspection_full.txt"
with open(path, "w", encoding="utf-8") as f:
    for col in cols:
        f.write(f"\n{'='*80}\n")
        f.write(f"SUMMARY FOR: {col}\n")
        f.write(f"{'='*80}\n")
        
        cleaned_mask = df[col].notna() & (df[col].astype(str).str.strip() != "")
        langs = df.loc[cleaned_mask, f"{col}_lang"]
        
        counts = langs.value_counts().reindex(["kh", "mixed", "en", "other"], fill_value=0)
        perc = (counts / counts.sum() * 100).round(1)
        summary = pd.DataFrame({
            "count": counts,
            "percent (%)": perc
        })
        
        translation_needed = counts["kh"] + counts["mixed"]
        translation_pct = round((translation_needed / counts.sum()) * 100, 1) if counts.sum() > 0 else 0
        
        f.write(summary.to_string())
        f.write(f"\n\nTotal needing translation (kh + mixed): {translation_needed} ({translation_pct}%)\n\n")
    
    f.write(f"\n{'='*80}\n")
    f.write("AGGREGATE LANGUAGE SUMMARY (BOTH TEXT COLUMNS)\n")
    f.write(f"{'='*80}\n")
    f.write(agg_summary.to_string())
    f.write(f"\n\nTotal needing translation (kh + mixed): {agg_translation_needed} ({agg_translation_pct}%)\n")
    f.write(f"Total Khmer only: {agg_counts['kh']}\n")
    f.write(f"Total Mixed: {agg_counts['mixed']}\n")
    f.write(f"Total Other: {agg_counts['other']}\n")
    
    for col in cols:
        for lang_type in ["kh", "mixed", "other"]:
            f.write(f"\n\n{'='*80}\n")
            f.write(f"{col} → {lang_type.upper()}\n")
            f.write(f"{'='*80}\n\n")
            
            subset = df[df[f"{col}_lang"] == lang_type][col]
            subset = subset.dropna().astype(str).str.strip()
            subset = subset[subset != ""]
            
            if len(subset) == 0:
                f.write("No responses found.\n")
            else:
                for i, val in enumerate(subset, 1):
                    f.write(f"{i}. {val}\n\n")

print("\nSaved full inspection to: language_inspection_full.txt")


SUMMARY FOR: open_strengths
                     count  percent (%)
open_strengths_lang                    
kh                      14          3.4
mixed                    2          0.5
en                     388         95.1
other                    4          1.0

Total needing translation (kh + mixed): 16 (3.9%)

SUMMARY FOR: open_challenges_suggestions
                                  count  percent (%)
open_challenges_suggestions_lang                    
kh                                   14          3.4
mixed                                 3          0.7
en                                  388         94.9
other                                 4          1.0

Total needing translation (kh + mixed): 17 (4.2%)

AGGREGATE LANGUAGE SUMMARY (BOTH TEXT COLUMNS)
       count  percent (%)
kh        28          3.4
mixed      5          0.6
en       776         95.0
other      8          1.0

Total needing translation (kh + mixed): 33 (4.0%)
Total Khmer only: 28
Total Mixed: 5
Tota

## Two translation approaches were evaluated: an open-source multilingual model (NLLB) and a Google-based translation tool. A comparative analysis showed that the Google-based method produced more fluent and semantically accurate translations.

## Approach 1: NLLB (Open-source translation) (used for comparison only) NLLB: No Language Left Behind is a multilingual translation model.

NLLB is a multilingual translation model in Hugging Face Transformers, supports 200+ languages, and the Hugging Face docs show the recommended usage pattern with src_lang and a forced target language token. The model card for facebook/nllb-200-distilled-600M is the main practical reference, and the original NLLB paper is No Language Left Behind: Scaling Human-Centered Machine Translation. https://huggingface.co/docs/transformers/model_doc/nllb?utm_source=chatgpt.com

In [6]:
import re
import gc
import time
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from blended_learning.utils.decorator import execution_time

# =========================================================
# 1. CONFIG
# =========================================================
MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "khm_Khmr"
TGT_LANG = "eng_Latn"

TEXT_COLUMNS = [
    "open_strengths",
    "open_challenges_suggestions"
]

BATCH_SIZE = 4
MAX_LENGTH = 256


# =========================================================
# 2. MODEL INITIALIZATION
# =========================================================
setup_start = time.perf_counter()

tokenizer = globals().get("tokenizer")
model = globals().get("model")

if tokenizer is None:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if model is None:
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model.eval()

setup_duration = time.perf_counter() - setup_start
print(f"⏱️ Model loaded in {setup_duration:.2f}s")


# =========================================================
# 3. LANGUAGE DETECTION
# =========================================================
KHMER_REGEX = re.compile(r"[\u1780-\u17FF]")
EN_REGEX = re.compile(r"[A-Za-z]")


def detect_language(text: str) -> str:
    if pd.isna(text):
        return "empty"

    text = str(text).strip()

    if not text:
        return "empty"

    has_kh = bool(KHMER_REGEX.search(text))
    has_en = bool(EN_REGEX.search(text))

    if has_kh and has_en:
        return "mixed"
    elif has_kh:
        return "kh"
    elif has_en:
        return "en"
    else:
        return "other"


# =========================================================
# 4. TRANSLATION FUNCTION
# =========================================================
@torch.no_grad()
def translate_batch(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    outputs_all = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = [str(x).strip() for x in texts[i:i + batch_size]]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
            src_lang=SRC_LANG,
        )

        generated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
            max_new_tokens=max_length,
            num_beams=4,
        )

        decoded = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True
        )

        outputs_all.extend(decoded)

        del inputs, generated
        gc.collect()

    return outputs_all


# =========================================================
# 5. ADD LANGUAGE LABELS
# =========================================================
for col in TEXT_COLUMNS:
    lang_col = f"{col}_lang"

    if lang_col not in df.columns:
        df[lang_col] = df[col].apply(detect_language)


# =========================================================
# 6. TRANSLATION PIPELINE
# =========================================================
@execution_time
def run_translation_pipeline(df, text_columns):
    for col in text_columns:
        lang_col = f"{col}_lang"
        trans_col = f"{col}_nllb_translated"

        print(f"\nProcessing: {col}")

        if trans_col not in df.columns:
            df[trans_col] = df[col]

        mask = (
            df[lang_col].isin(["kh", "mixed"])
            & df[col].notna()
            & (
                df[trans_col].isna()
                | df[trans_col].astype(str).str.strip().eq("")
                | (df[trans_col] == df[col])
            )
        )

        texts = df.loc[mask, col].astype(str).tolist()

        print(f"Rows to translate: {len(texts)}")

        if texts:
            translated = translate_batch(texts)
            df.loc[mask, trans_col] = translated
        else:
            print("No new translations.")

    return df


df = run_translation_pipeline(df, TEXT_COLUMNS)


# =========================================================
# 7. DEFINE NLP DATASET COLUMNS
# =========================================================
def get_nlp_columns(text_columns: list[str], df: pd.DataFrame) -> list[str]:
    """
    Build NLP dataset columns:
    - raw text
    - language detection
    - translated text
    """
    cols = []

    for c in text_columns:
        cols.extend([
            c,
            f"{c}_lang",
            f"{c}_nllb_translated"
        ])

    return [c for c in cols if c in df.columns]


nlp_columns = get_nlp_columns(TEXT_COLUMNS, df)


# =========================================================
# 8. SAVE NLP DATASET ONLY
# =========================================================
output_path = settings.root / "data" / "processed" / "translation_nllb.csv"

df[nlp_columns].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved NLP dataset → {output_path}")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

⏱️ Model loaded in 9.88s

Processing: open_strengths
Rows to translate: 16


Translating:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Processing: open_challenges_suggestions
Rows to translate: 17


Translating:   0%|          | 0/5 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

run_translation_pipeline executed in 3.16 min (9.46 sec)
Saved NLP dataset → C:\Users\Tepy\Documents\tepy\Final Internship Docs\blended_learning\data\processed\translation_nllb.csv


## Approach 2: Google-based translation (deep-translator) (selected for final preprocessing)

In [7]:
# =========================================================
# DEEP TRANSLATOR TRANSLATION FOR KHMER AND MIXED RESPONSES
# =========================================================

import time
import pandas as pd
from deep_translator import GoogleTranslator
from tqdm.auto import tqdm
from blended_learning.utils.decorator import execution_time

# =========================================================
# 1. CONFIG
# =========================================================
TEXT_COLUMNS = [
    "open_strengths",
    "open_challenges_suggestions"
]

SOURCE_LANG = "km"
TARGET_LANG = "en"

# =========================================================
# 2. TRANSLATOR INITIALIZATION
# =========================================================
setup_start = time.perf_counter()

translator = GoogleTranslator(
    source=SOURCE_LANG,
    target=TARGET_LANG
)

setup_duration = time.perf_counter() - setup_start
print(f"⏱️ Deep Translator initialized in {setup_duration:.2f}s")


# =========================================================
# 3. TRANSLATION FUNCTION
# =========================================================
def translate_deep(text):
    try:
        if pd.isna(text) or str(text).strip() == "":
            return text

        return translator.translate(str(text).strip())

    except Exception as e:
        print(f"Translation failed: {text} | Error: {e}")
        return text


# =========================================================
# 4. TRANSLATION PIPELINE
# =========================================================
@execution_time
def run_deep_translation_pipeline(df, text_columns):
    for col in text_columns:
        col_start = time.perf_counter()

        lang_col = f"{col}_lang"
        deep_col = f"{col}_deep_translated"

        print(f"\nProcessing: {col}")

        if deep_col not in df.columns:
            df[deep_col] = df[col]

        mask = (
            df[lang_col].isin(["kh", "mixed"])
            & df[col].notna()
            & (df[col].astype(str).str.strip() != "")
            & (
                df[deep_col].isna()
                | df[deep_col].astype(str).str.strip().eq("")
                | (df[deep_col] == df[col])
            )
        )

        texts = df.loc[mask, col].astype(str).tolist()

        print(f"Rows to translate: {len(texts)}")

        if texts:
            translated = [
                translate_deep(text)
                for text in tqdm(texts, desc=f"Translating {col}")
            ]

            df.loc[mask, deep_col] = translated
        else:
            print("No new translations.")

        col_duration = time.perf_counter() - col_start
        print(f"Processing time for {col}: {col_duration:.2f}s")

    return df


df = run_deep_translation_pipeline(df, TEXT_COLUMNS)


# =========================================================
# 5. DEFINE NLP DATASET COLUMNS
# =========================================================
def get_deep_translation_columns(text_columns, df):
    cols = []

    for c in text_columns:
        cols.extend([
            c,
            f"{c}_lang",
            f"{c}_deep_translated"
        ])

    return [c for c in cols if c in df.columns]


deep_columns = get_deep_translation_columns(TEXT_COLUMNS, df)


# =========================================================
# 6. SAVE DEEP TRANSLATOR DATASET ONLY
# =========================================================
output_path = settings.root / "data" / "processed" / "translation_deeptranslator.csv"

df[deep_columns].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved Deep Translator dataset → {output_path}")

⏱️ Deep Translator initialized in 0.00s

Processing: open_strengths
Rows to translate: 16


Translating open_strengths:   0%|          | 0/16 [00:00<?, ?it/s]

Processing time for open_strengths: 13.67s

Processing: open_challenges_suggestions
Rows to translate: 17


Translating open_challenges_suggestions:   0%|          | 0/17 [00:00<?, ?it/s]

Processing time for open_challenges_suggestions: 14.01s
run_deep_translation_pipeline executed in 0.46 min (27.67 sec)
Saved Deep Translator dataset → C:\Users\Tepy\Documents\tepy\Final Internship Docs\blended_learning\data\processed\translation_deeptranslator.csv


In [8]:
df

,gender,age,is_itc_student,itc_campus,province,itc_student_id,education_level,department,faculty,academic_year,...,gmm_entropy,gmm_prob_c1,gmm_prob_c2,sat_bin,open_strengths_lang,open_challenges_suggestions_lang,open_strengths_nllb_translated,open_challenges_suggestions_nllb_translated,open_strengths_deep_translated,open_challenges_suggestions_deep_translated
0,Male,36,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210528,Year5 - Final Year,GIC,NaN,2021–2022,...,-1.000089e-12,1.000000e+00,8.312872e-25,High (4-5),empty,empty,NaN,NaN,NaN,NaN
1,Male,23,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210686,Year5 - Final Year,GIC,NaN,2025–2026,...,5.010925e-03,9.994054e-01,5.946051e-04,High (4-5),en,en,Nothing,Nothing,Nothing,Nothing
2,Male,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20241146,Year2 - Sophomore,Foundation Year,NaN,2024–2025,...,3.937647e-04,3.496683e-05,9.999650e-01,High (4-5),en,en,"Very good, excellent","No big challenge, i’m the best","Very good, excellent","No big challenge, i’m the best"
3,Female,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240609,Year2 - Sophomore,GIC,NaN,2024–2025,...,5.971001e-03,7.256908e-04,9.992743e-01,Medium (3),en,en,Will try hard,Lack of self-discipline,Will try hard,Lack of self-discipline
4,Female,18,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240542,Year2 - Sophomore,GIC,NaN,2024–2025,...,5.097493e-02,8.919417e-03,9.910806e-01,Medium (3),en,en,getting more experience,Discipline on daily studying,getting more experience,Discipline on daily studying
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,Female,22,False,NaN,Phnom Penh,NaN,Other,NaN,Other,2024–2025,...,6.725031e-06,4.294172e-07,9.999996e-01,High (4-5),en,en,Blended learning is good because it gives flex...,"Big challenges are time management, unclear in...",Blended learning is good because it gives flex...,"Big challenges are time management, unclear in..."
416,Female,27,False,NaN,Phnom Penh,NaN,Other,NaN,Other,2019–2020,...,2.195298e-04,1.844694e-05,9.999816e-01,High (4-5),empty,empty,NaN,NaN,NaN,NaN
417,Female,24,False,NaN,Phnom Penh,NaN,Other,NaN,Faculty of Social Sciences,2021–2022,...,1.238122e-03,1.238579e-04,9.998761e-01,High (4-5),en,en,Flexible and self-discipline,The school should have focal points person to ...,Flexible and self-discipline,The school should have focal points person to ...
418,Female,25,False,NaN,Phnom Penh,NaN,Other,NaN,Other,2019–2020,...,5.862005e-03,7.106321e-04,9.992894e-01,Medium (3),en,en,"I feel comfort, no need to see many people",lost confident,"I feel comfort, no need to see many people",lost confident


# CREATE VALIDATION COLUMNS IN df

In [9]:

TEXT_COLUMNS = [
    "open_strengths",
    "open_challenges_suggestions"
]

for col in TEXT_COLUMNS:
   

    df[f"{col}_nllb_quality"] = ""     # correct / minor_error / incorrect
    df[f"{col}_deep_quality"] = ""     # correct / minor_error / incorrect
    df[f"{col}_manual_validated_translation"] = ""


print("Validation columns created.")

display(df.head())

Validation columns created.


,gender,age,is_itc_student,itc_campus,province,itc_student_id,education_level,department,faculty,academic_year,...,open_strengths_nllb_translated,open_challenges_suggestions_nllb_translated,open_strengths_deep_translated,open_challenges_suggestions_deep_translated,open_strengths_nllb_quality,open_strengths_deep_quality,open_strengths_manual_validated_translation,open_challenges_suggestions_nllb_quality,open_challenges_suggestions_deep_quality,open_challenges_suggestions_manual_validated_translation
0,Male,36,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210528,Year5 - Final Year,GIC,NaN,2021–2022,...,NaN,NaN,NaN,NaN,,,,,,
1,Male,23,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210686,Year5 - Final Year,GIC,NaN,2025–2026,...,Nothing,Nothing,Nothing,Nothing,,,,,,
2,Male,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20241146,Year2 - Sophomore,Foundation Year,NaN,2024–2025,...,"Very good, excellent","No big challenge, i’m the best","Very good, excellent","No big challenge, i’m the best",,,,,,
3,Female,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240609,Year2 - Sophomore,GIC,NaN,2024–2025,...,Will try hard,Lack of self-discipline,Will try hard,Lack of self-discipline,,,,,,
4,Female,18,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240542,Year2 - Sophomore,GIC,NaN,2024–2025,...,getting more experience,Discipline on daily studying,getting more experience,Discipline on daily studying,,,,,,


In [10]:
# =========================================================
# SAVE df TO EXCEL
# =========================================================

output_path = settings.root / "data" / "processed" / "translation_validation_file.xlsx"

df.to_excel(
    output_path,
    index=False,
    engine="openpyxl"
)

print(f"DataFrame saved to Excel: {output_path}")
print(f"Shape: {df.shape}")

DataFrame saved to Excel: C:\Users\Tepy\Documents\tepy\Final Internship Docs\blended_learning\data\processed\translation_validation_file.xlsx
Shape: (420, 107)


In [11]:
# df_n = df[nlp_columns]

In [ ]:
# df_n['challenges_valid'] = df_n['open_strengths']

NameError: name 'df_n' is not defined

In [ ]:
# df_n

,open_strengths,open_strengths_lang,open_strengths_nllb_translated,open_challenges_suggestions,open_challenges_suggestions_lang,open_challenges_suggestions_nllb_translated,challenges_valid
0,NaN,empty,NaN,NaN,empty,NaN,NaN
1,Nothing,en,Nothing,Nothing,en,Nothing,Nothing
2,"Very good, excellent",en,"Very good, excellent","No big challenge, i’m the best",en,"No big challenge, i’m the best","Very good, excellent"
3,Will try hard,en,Will try hard,Lack of self-discipline,en,Lack of self-discipline,Will try hard
4,getting more experience,en,getting more experience,Discipline on daily studying,en,Discipline on daily studying,getting more experience
...,...,...,...,...,...,...,...
415,Blended learning is good because it gives flex...,en,Blended learning is good because it gives flex...,"Big challenges are time management, unclear in...",en,"Big challenges are time management, unclear in...",Blended learning is good because it gives flex...
416,NaN,empty,NaN,NaN,empty,NaN,NaN
417,Flexible and self-discipline,en,Flexible and self-discipline,The school should have focal points person to ...,en,The school should have focal points person to ...,Flexible and self-discipline
418,"I feel comfort, no need to see many people",en,"I feel comfort, no need to see many people",lost confident,en,lost confident,"I feel comfort, no need to see many people"


In [ ]:
# len(df_n[df_n["open_strengths_lang"].isin(["kh", "mixed"])])

16

In [ ]:
# df_n['open_strengths_validation'] = 'Accuracy'

In [ ]:
# df_n[df_n["open_strengths_lang"].isin(["kh", "mixed"])]

,open_strengths,open_strengths_lang,open_strengths_nllb_translated,open_challenges_suggestions,open_challenges_suggestions_lang,open_challenges_suggestions_nllb_translated,challenges_valid,open_strengths_validation
29,ខ្ញុំថាវាមានប្រយោជន៍នៅពេលគ្រូមានការរវល់អាចបង្រ...,kh,I think it's helpful when there's a problem. T...,ខ្ញុំជួបបញ្ហាអ៊ីនធឺណេត,kh,I've had problems on the internet.,ខ្ញុំថាវាមានប្រយោជន៍នៅពេលគ្រូមានការរវល់អាចបង្រ...,Accuracy
48,យល់ដឹងពីការប្រើប្រាស់បច្ចេកវិទ្យា ប្រព័ន្ធឌីជីថល,kh,Knowledge of the use of digital technologies,បញ្ហាតភ្ជាប់អ៊ីនធឺណិត ពិបាកយល់ខ្លឹមសារដោយមិនម...,kh,"Having difficulty connecting to the Internet, ...",យល់ដឹងពីការប្រើប្រាស់បច្ចេកវិទ្យា ប្រព័ន្ធឌីជីថល,Accuracy
90,ល្អទាំងអស់,kh,It's all good.,គ្រប់យ៉ាងល្អ,kh,It's all good.,ល្អទាំងអស់,Accuracy
92,តាមទាម់មេរៀនបានល្អ នៅពេលអវត្តមាននៅក្នុងម៉ោងសិស...,kh,"As per the teachings, it is good when there is...",អ៊ីនធឺណិត ត្រូវចំណាយច្រើនទៅលើអ៊ីនធឺណិត,kh,The Internet needs to spend a lot on the Inter...,តាមទាម់មេរៀនបានល្អ នៅពេលអវត្តមាននៅក្នុងម៉ោងសិស...,Accuracy
106,អាចអោយសិស្សឬនិស្សិតសិក្សាខ្លួនឯងគ្រប់ទីកន្លែង ...,kh,can allow the student or student to study hims...,សិស្សឬនិស្សិតអាចបើការសិក្សាតាមonlineចោលឬមិនយកច...,mixed,The student or the student can either leave th...,អាចអោយសិស្សឬនិស្សិតសិក្សាខ្លួនឯងគ្រប់ទីកន្លែង ...,Accuracy
131,ចំណេញពេលវេលា និង កាត់បន្ថយចំណាយ,kh,The excess of time and the reduction of expend...,មានពេលវេលាកំណត់ក្នុងការមើលមេរៀន,kh,There's a time to look at the classroom.,ចំណេញពេលវេលា និង កាត់បន្ថយចំណាយ,Accuracy
136,អាចរក្សាទុកវីដេអូ និងអាចមើលវីដេអូឡើងវិញបាន,kh,Can save the video and can view the video up a...,ជួយបង្កើនសេវាអ៊ីនធើណេតនៅសាលា,kh,Help improve the internet service at school,អាចរក្សាទុកវីដេអូ និងអាចមើលវីដេអូឡើងវិញបាន,Accuracy
137,អាចឲ្យសិស្សស្វ័យសិក្សាដោយខ្លួនឯងពេលសិស្សរៀនផ្ទ...,kh,Can make the student learn on his own when the...,ពេលកូនសិស្ស submit assignments លោកគ្រូអ្នកគ្រូ...,mixed,"When a student submits assignments, the teache...",អាចឲ្យសិស្សស្វ័យសិក្សាដោយខ្លួនឯងពេលសិស្សរៀនផ្ទ...,Accuracy
143,ចេះប្រើប្រព័ន្ធឬបច្ចេកទេសថ្មីៗ,kh,know how to use the latest systems or techniques,គ្មានទេ,kh,There is none.,ចេះប្រើប្រព័ន្ធឬបច្ចេកទេសថ្មីៗ,Accuracy
171,តាមគំនិតរបស់ខ្ញុំ ចំនុចល្អនៃការសិក្សាបែបចម្រុះ...,kh,"In my opinion, the best thing about studying i...",មិនមានទេ,kh,There isn't.,តាមគំនិតរបស់ខ្ញុំ ចំនុចល្អនៃការសិក្សាបែបចម្រុះ...,Accuracy


# LOAD MANUAL TRANSLATION VALIDATION FILE

In [18]:

TEXT_COLUMNS = [
    "open_strengths",
    "open_challenges_suggestions"
]

validated_path = settings.root / "data" / "processed" / "translation_validation_file.xlsx"

validated_df = pd.read_excel(validated_path)

print(f"Validated file loaded: {validated_df.shape}")
display(validated_df.head())

Validated file loaded: (420, 107)


,gender,age,is_itc_student,itc_campus,province,itc_student_id,education_level,department,faculty,academic_year,...,open_strengths_nllb_translated,open_challenges_suggestions_nllb_translated,open_strengths_deep_translated,open_challenges_suggestions_deep_translated,open_strengths_nllb_quality,open_strengths_deep_quality,open_strengths_manual_validated_translation,open_challenges_suggestions_nllb_quality,open_challenges_suggestions_deep_quality,open_challenges_suggestions_manual_validated_translation
0,Male,36,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210528,Year5 - Final Year,GIC,NaN,2021–2022,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Male,23,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210686,Year5 - Final Year,GIC,NaN,2025–2026,...,Nothing,Nothing,Nothing,Nothing,NaN,NaN,NaN,NaN,NaN,NaN
2,Male,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20241146,Year2 - Sophomore,Foundation Year,NaN,2024–2025,...,"Very good, excellent","No big challenge, i’m the best","Very good, excellent","No big challenge, i’m the best",NaN,NaN,NaN,NaN,NaN,NaN
3,Female,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240609,Year2 - Sophomore,GIC,NaN,2024–2025,...,Will try hard,Lack of self-discipline,Will try hard,Lack of self-discipline,NaN,NaN,NaN,NaN,NaN,NaN
4,Female,18,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240542,Year2 - Sophomore,GIC,NaN,2024–2025,...,getting more experience,Discipline on daily studying,getting more experience,Discipline on daily studying,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
# =========================================================
# TRANSLATION ACCURACY AND ACCEPTABLE RATE REPORT
# =========================================================

def clean_quality_label(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )


def translation_validation_report(series):
    series = clean_quality_label(series)

    valid_labels = [
        "accurate",
        "partially_accurate",
        "inaccurate"
    ]

    total = series.isin(valid_labels).sum()

    accurate = (series == "accurate").sum()
    partially_accurate = (series == "partially_accurate").sum()
    inaccurate = (series == "inaccurate").sum()

    accuracy = accurate / total * 100 if total > 0 else 0
    acceptable_rate = (accurate + partially_accurate) / total * 100 if total > 0 else 0

    return pd.Series({
        "total_checked": total,
        "accurate": accurate,
        "partially_accurate": partially_accurate,
        "inaccurate": inaccurate,
        "accuracy_%": round(accuracy, 2),
        "acceptable_rate_%": round(acceptable_rate, 2)
    })

In [20]:
# =========================================================
# REPORT BY TEXT COLUMN AND TRANSLATION METHOD
# =========================================================

reports = []

for col in TEXT_COLUMNS:
    nllb_quality_col = f"{col}_nllb_quality"
    deep_quality_col = f"{col}_deep_quality"

    nllb_report = translation_validation_report(validated_df[nllb_quality_col])
    nllb_report["text_column"] = col
    nllb_report["method"] = "NLLB"

    deep_report = translation_validation_report(validated_df[deep_quality_col])
    deep_report["text_column"] = col
    deep_report["method"] = "Deep Translator"

    reports.append(nllb_report)
    reports.append(deep_report)

translation_report = pd.DataFrame(reports)

translation_report = translation_report[
    [
        "text_column",
        "method",
        "total_checked",
        "accurate",
        "partially_accurate",
        "inaccurate",
        "accuracy_%",
        "acceptable_rate_%"
    ]
]

display(translation_report)

,text_column,method,total_checked,accurate,partially_accurate,inaccurate,accuracy_%,acceptable_rate_%
0,open_strengths,NLLB,17.0,7.0,8.0,2.0,41.18,88.24
1,open_strengths,Deep Translator,17.0,12.0,5.0,0.0,70.59,100.00
2,open_challenges_suggestions,NLLB,17.0,7.0,8.0,2.0,41.18,88.24
3,open_challenges_suggestions,Deep Translator,17.0,12.0,5.0,0.0,70.59,100.00


In [21]:
# =========================================================
# OVERALL TRANSLATION REPORT
# BOTH TEXT COLUMNS COMBINED
# =========================================================

nllb_all_quality = pd.concat(
    [validated_df[f"{col}_nllb_quality"] for col in TEXT_COLUMNS],
    ignore_index=True
)

deep_all_quality = pd.concat(
    [validated_df[f"{col}_deep_quality"] for col in TEXT_COLUMNS],
    ignore_index=True
)

overall_translation_report = pd.DataFrame({
    "NLLB": translation_validation_report(nllb_all_quality),
    "Deep Translator": translation_validation_report(deep_all_quality)
}).T

display(overall_translation_report)

,total_checked,accurate,partially_accurate,inaccurate,accuracy_%,acceptable_rate_%
NLLB,34.0,14.0,16.0,4.0,41.18,88.24
Deep Translator,34.0,24.0,10.0,0.0,70.59,100.00


In [23]:
# =========================================================
# SAVE TRANSLATION ACCURACY REPORT
# =========================================================

report_path = settings.root / "data" / "processed" / "translation_accuracy_report.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    translation_report.to_excel(
        writer,
        sheet_name="By_Text_Column",
        index=False
    )

    overall_translation_report.to_excel(
        writer,
        sheet_name="Overall"
    )

print(f"Translation accuracy report saved to: {report_path}")

Translation accuracy report saved to: C:\Users\Tepy\Documents\tepy\Final Internship Docs\blended_learning\data\processed\translation_accuracy_report.xlsx


# Load the validated Excel file for the NLP analysis

In [24]:

import pandas as pd
import numpy as np
import re

TEXT_COLUMNS = [
    "open_strengths",
    "open_challenges_suggestions"
]

validated_path = settings.root / "data" / "processed" / "translation_validation_file.xlsx"

df = pd.read_excel(validated_path)

print(f"Validated dataset loaded: {df.shape}")
display(df.head())

Validated dataset loaded: (420, 107)


,gender,age,is_itc_student,itc_campus,province,itc_student_id,education_level,department,faculty,academic_year,...,open_strengths_nllb_translated,open_challenges_suggestions_nllb_translated,open_strengths_deep_translated,open_challenges_suggestions_deep_translated,open_strengths_nllb_quality,open_strengths_deep_quality,open_strengths_manual_validated_translation,open_challenges_suggestions_nllb_quality,open_challenges_suggestions_deep_quality,open_challenges_suggestions_manual_validated_translation
0,Male,36,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210528,Year5 - Final Year,GIC,NaN,2021–2022,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Male,23,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210686,Year5 - Final Year,GIC,NaN,2025–2026,...,Nothing,Nothing,Nothing,Nothing,NaN,NaN,NaN,NaN,NaN,NaN
2,Male,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20241146,Year2 - Sophomore,Foundation Year,NaN,2024–2025,...,"Very good, excellent","No big challenge, i’m the best","Very good, excellent","No big challenge, i’m the best",NaN,NaN,NaN,NaN,NaN,NaN
3,Female,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240609,Year2 - Sophomore,GIC,NaN,2024–2025,...,Will try hard,Lack of self-discipline,Will try hard,Lack of self-discipline,NaN,NaN,NaN,NaN,NaN,NaN
4,Female,18,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240542,Year2 - Sophomore,GIC,NaN,2024–2025,...,getting more experience,Discipline on daily studying,getting more experience,Discipline on daily studying,NaN,NaN,NaN,NaN,NaN,NaN


# Create final English columns for NLP

In [26]:


for col in TEXT_COLUMNS:
    deep_col = f"{col}_deep_translated"
    manual_col = f"{col}_manual_validated_translation"
    final_col = f"{col}_final_en"

    # Start from Deep Translator column
    # For English responses, this should already be the original English text
    df[final_col] = df[deep_col]

    # Replace with manual validated translation when available
    mask_manual = (
        df[manual_col].notna()
        & (df[manual_col].astype(str).str.strip() != "")
    )

    df.loc[mask_manual, final_col] = df.loc[mask_manual, manual_col]

print("Final English NLP columns created.")

display(df[
    [
        "open_strengths",
        "open_strengths_lang",
        "open_strengths_final_en",
        "open_challenges_suggestions",
        "open_challenges_suggestions_lang",
        "open_challenges_suggestions_final_en"
    ]
].head())

Final English NLP columns created.


,open_strengths,open_strengths_lang,open_strengths_final_en,open_challenges_suggestions,open_challenges_suggestions_lang,open_challenges_suggestions_final_en
0,NaN,empty,NaN,NaN,empty,NaN
1,Nothing,en,Nothing,Nothing,en,Nothing
2,"Very good, excellent",en,"Very good, excellent","No big challenge, i’m the best",en,"No big challenge, i’m the best"
3,Will try hard,en,Will try hard,Lack of self-discipline,en,Lack of self-discipline
4,getting more experience,en,getting more experience,Discipline on daily studying,en,Discipline on daily studying


# Save the NLP-ready dataset

In [27]:


output_excel = settings.root / "data" / "processed" / "nlp_ready_manual_validated.xlsx"
output_csv = settings.root / "data" / "processed" / "nlp_ready_manual_validated.csv"

df.to_excel(
    output_excel,
    index=False,
    engine="openpyxl"
)

df.to_csv(
    output_csv,
    index=False,
    encoding="utf-8-sig"
)

print(f"NLP-ready Excel saved to: {output_excel}")
print(f"NLP-ready CSV saved to: {output_csv}")
print(f"Final shape: {df.shape}")

NLP-ready Excel saved to: C:\Users\Tepy\Documents\tepy\Final Internship Docs\blended_learning\data\processed\nlp_ready_manual_validated.xlsx
NLP-ready CSV saved to: C:\Users\Tepy\Documents\tepy\Final Internship Docs\blended_learning\data\processed\nlp_ready_manual_validated.csv
Final shape: (420, 109)
